# 17-тарау. Гүлдің түрін анықтаймыз

*«Машина қалай үйренеді?» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 17.1. Жобаның мақсаты

## 17.2. 1-қадам. Деректерді жүктеу және қарау

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df = iris.frame
df.columns = ["тост_ұзын", "тост_ені", "күлте_ұзын", "күлте_ені", "түр"]
print(df.sample(5, random_state=1))

In [ ]:
print(df.groupby("түр").mean().round(2))

## 17.3. 2-қадам. Визуализация

## 17.4. 3-қадам. Модельдерді салыстыру

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

X = df.drop(columns="түр")
y = df["түр"]

scaled = lambda model: make_pipeline(StandardScaler(), model)
models = {
    "kNN (k=5)": scaled(KNeighborsClassifier(5)),
    "Логистикалық регрессия": scaled(LogisticRegression(max_iter=1000)),
    "Шешім ағашы": DecisionTreeClassifier(max_depth=3, random_state=0),
    "Кездейсоқ орман": RandomForestClassifier(n_estimators=100,
                                              random_state=0),
    "SVM": scaled(SVC()),
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5)
    results[name] = scores
    print(f"{name:24s} {scores.mean():.3f} ± {scores.std():.3f}")

## 17.5. 4-қадам. Таңдалған модельді тексеру

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

best = models["SVM"].fit(X_train, y_train)
print(classification_report(y_test, best.predict(X_test),
                            target_names=iris.target_names))

## 17.6. 5-қадам. Модельді сақтау және қолдану

In [ ]:
import joblib

joblib.dump(best, "iris_model.joblib")          # сақтау
loaded = joblib.load("iris_model.joblib")       # жүктеу

new_flowers = pd.DataFrame([[5.0, 3.4, 1.5, 0.2],
                            [6.3, 2.9, 5.6, 1.8],
                            [5.9, 3.0, 4.2, 1.3]], columns=X.columns)
pred = loaded.predict(new_flowers)
print(iris.target_names[pred])